# 05. Protein Language Model (PLM) Embeddings
This notebook loads the generative Protein Language Model (ProGen2) and extracts contextual sequence representation embeddings ($\mathbf{z}_q^P = f_{\theta_P}(\mathcal{T}_P(\mathbf{a}_q^P))$) for protein sequences as specified in Section 5.2 and Section 9 of the Mathematical Model.

In [ ]:
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

PROJECT_ROOT = Path('.').resolve().parents[0]
PROCESSED_PROTEIN_DIR = PROJECT_ROOT / 'data' / 'processed' / 'sequences' / 'protein'
EMBEDDINGS_DIR = PROJECT_ROOT / 'data' / 'processed' / 'embeddings'
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Execution Device: {DEVICE}')

## Loading PLM and Extracting Embeddings ($\mathbf{z}_q^P$)

In [ ]:
model_name = 'Salesforce/progen2-small'
print(f'Loading Protein Language Model: {model_name}...')

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, output_hidden_states=True)
model.to(DEVICE)
model.eval()

# Example protein sequence extraction
sample_sequence = 'MKWVTFISLLFLFSSAYSRV'
inputs = tokenizer(sample_sequence, return_tensors='pt').to(DEVICE)

with torch.no_grad():
    outputs = model(**inputs)
    # Extract the last hidden state for z_q^P
    hidden_states = outputs.hidden_states[-1]
    z_q_p = hidden_states.mean(dim=1).cpu().numpy()

print(f'Extracted Protein Embedding shape (z_q^P): {z_q_p.shape}')